# 16.1 使用字符RNN生成莎士比亚文本

## 16.1.1 创建训练数据集

In [4]:


# 下载数据
from tensorflow import keras



shakespeare_url = "https://homl.info/shakespeare"
filepath = keras.utils.get_file("shakespeare.txt", shakespeare_url)
with open(filepath) as f:
    shakespeare_text = f.read()


In [5]:
# 将每个字符编码为整数
from tensorflow.keras.preprocessing.text import Tokenizer

# 字符级分词器
tokenizer = Tokenizer(char_level=True)
tokenizer.fit_on_texts([shakespeare_text])

现在，字符可以将一个句子（或句子列表）编码为字符ID列表并返回

In [6]:
tokenizer.texts_to_sequences(['First'])

[[20, 6, 9, 8, 3]]

In [7]:
tokenizer.sequences_to_texts([[20,6,9,8,3]])

['f i r s t']

In [8]:
max_id = len(tokenizer.word_index)
dataset_size = tokenizer.document_count

In [9]:
# 对全文进行编码(减去一代表着从0到38的编码而不是1到39)
import  numpy as np
[encoded] = np.array(tokenizer.texts_to_sequences([shakespeare_text]))-1

## 16.1.2 如何拆分顺序数据集

In [10]:
# 使用文本的前90%作为训练集
import tensorflow as tf
train_size = dataset_size*90//100
dataset = tf.data.Dataset.from_tensor_slices(encoded[:train_size])

## 16.1.3 将顺序数据集切成多个窗口

In [11]:
# 创建短文本窗口的数据集
n_steps = 100# 输入序列的长度：100个字符
window_length = n_steps+1#窗口总长度：101个字符
dataset = dataset.window(window_length,shift = 1 ,drop_remainder=True)

In [12]:
# 展平数据集
dataset = dataset.flat_map(lambda window: window.batch(window_length))

In [13]:
# 批处理这些窗口并将输入与目标分开
batch_size = 32
dataset = dataset.shuffle(10000).batch(batch_size)
dataset = dataset.map(lambda windows:(windows[:,:-1],windows[:,1:]))

In [14]:
# 使用独热向量对每一个字符进行编码
dataset = dataset.map(
    lambda X_batch,y_batch: (tf.one_hot(X_batch,depth = max_id), y_batch)
)

In [15]:
# 添加预取
dataset = dataset.prefetch(1)

## 16.1.4 创建和训练Char-RNN模型

In [16]:
# 先算总样本数
total_samples = len(encoded) - window_length + 1
steps_per_epoch = total_samples // batch_size

model = keras.Sequential([
    keras.layers.GRU(256,return_sequences=True,input_shape=[None,max_id],dropout = 0.2,recurrent_dropout = 0.2),
    keras.layers.GRU(256,return_sequences=True,dropout = 0.2,recurrent_dropout = 0.2),
    keras.layers.TimeDistributed(keras.layers.Dense(max_id,activation="softmax"))
])
model.compile(loss = 'sparse_categorical_crossentropy', optimizer = 'adam')
history = model.fit(dataset,epochs = 80,steps_per_epoch = steps_per_epoch)

C:\Users\24677\anaconda3\envs\ml\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/80
34852/34852 ━━━━━━━━━━━━━━━━━━━━ 6s 161us/step
Epoch 2/80
34852/34852 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
Epoch 3/80
34852/34852 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
Epoch 4/80
34852/34852 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
Epoch 5/80
34852/34852 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
Epoch 6/80


C:\Users\24677\anaconda3\envs\ml\lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


34852/34852 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
Epoch 7/80
34852/34852 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
Epoch 8/80
34852/34852 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
Epoch 9/80
34852/34852 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
Epoch 10/80
34852/34852 ━━━━━━━━━━━━━━━━━━━━ 0s 2us/step
Epoch 11/80
34852/34852 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
Epoch 12/80
34852/34852 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
Epoch 13/80
34852/34852 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
Epoch 14/80
34852/34852 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
Epoch 15/80
34852/34852 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
Epoch 16/80
34852/34852 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
Epoch 17/80
34852/34852 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
Epoch 18/80
34852/34852 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
Epoch 19/80
34852/34852 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
Epoch 20/80
34852/34852 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
Epoch 21/80
34852/34852 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
Epoch 22/80
34852/34852 ━━━━━━━━━━━━━━━━━━━━ 0s 1us/step
Epoch 23/80
34852/34852 ━━━━━━━━━━━━━━━━━━━━ 0

## 16.1.5 使用Char-RNN模型

In [17]:
#预处理函数
def preprocess(texts):
    X = np.array(tokenizer.texts_to_sequences(texts))-1
    return tf.one_hot(X,max_id)

In [18]:
# 预测下一个字母
X_new = preprocess(['how are yo'])
Y_pred_proba = model.predict(X_new)
Y_pred = np.argmax(Y_pred_proba,axis=-1)
tokenizer.sequences_to_texts(Y_pred+1)[0][-1]

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 982ms/step


'r'

## 16.1.6 生成假莎士比亚文本

In [19]:
# 选择要添加到输入文本的下一个字符
def next_char(text,temperature = 1):
    X_new = preprocess([text])
    y_proba = model.predict(X_new)[0,-1:,:]
    rescaled_logits = tf.math.log(y_proba/temperature)
    char_id = tf.random.categorical(rescaled_logits,num_samples=1)+1
    return tokenizer.sequences_to_texts(char_id.numpy())[0]

In [20]:
# 获得下一个字符
def complete_text(text,n_chars = 50,temperature = 1):
    for _ in range(n_chars):
        text += next_char(text,temperature)
    return text


In [21]:
print(complete_text('t',temperature = 0.2))

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 772ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━

In [22]:
print(complete_text('w',temperature = 1))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━

In [23]:
print(complete_text('w',temperature = 2))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━

## 16.1.7 有状态的RNN

In [24]:
# 无状态RNN的批处理
dataset = tf.data.Dataset.from_tensor_slices(encoded[:train_size])
dataset = dataset.window(window_length,shift = n_steps ,drop_remainder=True)
dataset = dataset.flat_map(lambda window: window.batch(window_length))
dataset = dataset.batch(1)
dataset = dataset.map(lambda windows:(windows[:,:-1],windows[:,1:]))
dataset = dataset.map(
    lambda X_batch,y_batch: (tf.one_hot(X_batch,depth = max_id), y_batch)
)
dataset = dataset.prefetch(1)

In [25]:
# 创建有状态的RNN
from tensorflow import keras
model = keras.Sequential([
    keras.Input(batch_shape=(batch_size, None, max_id)),
    keras.layers.GRU(128,return_sequences = True,stateful = True,dropout= 0.2,recurrent_dropout=0.2),
    keras.layers.GRU(128,return_sequences = True,stateful = True,dropout= 0.2,recurrent_dropout=0.2),
    keras.layers.TimeDistributed(keras.layers.Dense(max_id,activation="softmax"))
])

In [26]:
from tensorflow import keras
model = keras.models.Sequential([
    keras.layers.GRU(128,return_sequences = True,stateful = True,dropout= 0.2,recurrent_dropout=0.2,batch_input_shape=[batch_size,None,max_id]),
    keras.layers.GRU(128,return_sequences = True,stateful = True,dropout= 0.2,recurrent_dropout=0.2),
    keras.layers.TimeDistributed(keras.layers.Dense(max_id,activation="softmax"))
])

ValueError: Unrecognized keyword arguments passed to GRU: {'batch_input_shape': [32, None, 39]}

In [27]:
#创建一个小的回调函数
class ResetStatesCallback(keras.callbacks.Callback):
    def on_epoch_begin(self, epoch, logs = None):
        if self.model is not None:  # 加上判断，避免报错
            self.model.reset_states()

In [28]:
# 编译并拟合模型
model.compile(loss = 'sparse_categorical_crossentropy', optimizer = 'adam')
model.fit(dataset,epochs = 50,callbacks = [ResetStatesCallback()])

AttributeError: 'Sequential' object has no attribute 'reset_states'

# 16.2 情感分析

In [29]:
# 加载数据
(X_train,y_train),(X_test,y_test) = tf.keras.datasets.imdb.load_data()
X_train[0][:10]

[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65]

In [30]:
# 编码
word_index = keras.datasets.imdb.get_word_index()
id_to_word = {id_+3:word for word,id_ in word_index.items()}
for id_,token in enumerate(('<pad>','<sos>','<unk>')):
    id_to_word[id_] = token

' '.join([id_to_word[id_] for id_ in X_train[0][:10]])

'<sos> this film was just brilliant casting location scenery story'

In [31]:
# 以文本的形式加载IMDb评论
import tensorflow_datasets as tfds

datasets,info  = tfds.load('imdb_reviews',as_supervised=True,with_info=True)
train_size = info.splits['train'].num_examples

In [32]:
# 写预处理函数
def preprocess(X_batch,y_batch):
    X_batch = tf.strings.substr(X_batch,0,300)
    X_batch = tf.strings.regex_replace(X_batch,b'<br\\s*/?>',b' ')
    X_batch = tf.strings.regex_replace(X_batch,b'[^a-zA-Z]',b' ')
    X_batch = tf.strings.split(X_batch)
    return X_batch.to_tensor(default_value=b'<pad>'),y_batch

In [33]:
# 构建词汇表
from collections import Counter
vocabulary = Counter()
for X_batch,y_batch in datasets['train'].batch(32).map(preprocess):
    for review in X_batch:
        vocabulary.update(list(review.numpy()))

In [34]:
vocabulary.most_common()[:3]

[(b'<pad>', 224494), (b'the', 61156), (b'a', 38569)]

In [35]:
# 截断词汇表
vocab_size = 10000
truncated_vocabulary = [
    word for word,count in vocabulary.most_common()[:vocab_size]
]

In [36]:
#创造一个查找表，将每个单词替换为其ID
words = tf.constant(truncated_vocabulary)
word_ids = tf.range(len(truncated_vocabulary),dtype=tf.int64)
vocab_init = tf.lookup.KeyValueTensorInitializer(words,word_ids)
num_oov_buckets = 1000
table = tf.lookup.StaticVocabularyTable(vocab_init,num_oov_buckets)

In [37]:
# 查找几个单词的ID
table.lookup(tf.constant([b'This movie was faaaaaantastic'.split()]))


<tf.Tensor: shape=(1, 4), dtype=int64, numpy=array([[   24,    12,    13, 10053]], dtype=int64)>

In [38]:
# 创建最终的训练表
def encode_words(X_batch,y_batch):
    return table.lookup(X_batch),y_batch

train_set = datasets['train'].batch(32).map(preprocess)
train_set = train_set.map(encode_words).prefetch(1)

In [39]:
# 创建模型并对其进行训练
embed_size = 128
model = keras.models.Sequential([
    keras.layers.Embedding(vocab_size+num_oov_buckets,embed_size,input_length=[None]),
    keras.layers.GRU(128,return_sequences=True),
    keras.layers.GRU(128),
    keras.layers.Dense(1,activation="sigmoid")
])

model.compile(loss = 'binary_crossentropy', optimizer = 'adam',metrics = ['accuracy'])
history = model.fit(train_set,epochs = 5)

Epoch 1/5


C:\Users\24677\anaconda3\envs\ml\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


782/782 ━━━━━━━━━━━━━━━━━━━━ 51s 60ms/step - accuracy: 0.5185 - loss: 0.6896
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 47s 60ms/step - accuracy: 0.7506 - loss: 0.4962
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 46s 59ms/step - accuracy: 0.8637 - loss: 0.3207
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 47s 60ms/step - accuracy: 0.9116 - loss: 0.2293
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 169s 216ms/step - accuracy: 0.9400 - loss: 0.1651


## 16.2.1 掩码屏蔽



### 基本思想
在自然语言处理中，模型需要忽略填充令牌（padding tokens），专注于有效数据。通过设置 `mask_zero=True`，可以让 Embedding 层自动生成掩码，下游层会忽略填充令牌（通常其 ID 为 0）。

### 实现方式
- Embedding 层会自动创建一个布尔掩码张量：`K.not_equal(inputs, 0)`（`K` 为 `keras.backend`）。掩码在单词 ID 为 0 的位置为 `False`，其余为 `True`。
- 建议始终将填充令牌编码为 0（即使它不是最频繁的单词）。

### 掩码的传播与处理
- 掩码张量会沿时间维度自动传播到所有支持掩码的后续层。
- 当循环层（如 GRU/LSTM）遇到被掩码的时间步长（掩码为 `False`）时，会复制前一时间步长的输出。
- 如果掩码传播到输出层（在输出序列的模型中），被掩码的时间步长不会对损失产生影响（损失为 0）。
- **注意**：LSTM 和 GRU 层基于 cuDNN 的 GPU 优化实现**不支持掩码**。使用掩码会导致这些层回退到较慢的默认实现。优化实现还要求使用多个超参数的默认值：`activation`, `recurrent_activation`, `recurrent_dropout`, `unroll`, `use_bias`, `reset_after`。

### 层对掩码的支持要求
- 支持掩码的层必须具有 `supports_masking = True` 属性，否则会引发异常。
- 所有循环层、`TimeDistributed` 层等均支持掩码。
- 自定义层若要支持掩码，需在构造函数中设置 `self.supports_masking = True`，并在 `call()` 方法中添加 `mask` 参数。

### 使用 `Masking` 层
如果模型不是从 Embedding 层开始，可以使用 `keras.layers.Masking` 层。它会将掩码设置为 `K.any(K.not_equal(inputs, 0), axis=-1)`，即当某个时间步长的最后一维全部为零时，该步长被屏蔽。



In [40]:
# 使用函数式API构建并手动处理掩码
K = keras.backend
inputs = keras.layers.Input(shape=[None])
mask = keras.layers.Lambda(lambda inputs:K.not_equal(inputs,0))(inputs)
z = keras.layers.Embedding(vocab_size+num_oov_buckets,embed_size)(inputs)
z = keras.layers.GRU(128,return_sequences=True)(z,mask=mask)
z = keras.layers.GRU(128)(z,mask=mask)
outputs = keras.layers.Dense(1,activation="sigmoid")(z)
model = keras.Model(inputs = [inputs],outputs = [outputs])

## 16.2.2 重用预训练的嵌入

In [41]:
import tensorflow_hub as hub
import tensorflow as tf
from tensorflow import keras

hub_layer = hub.KerasLayer(
    "https://tfhub.dev/google/tf2-preview/nnlm-en-dim50/1",
    dtype=tf.string,
    input_shape=[]
)

model = keras.Sequential([
    keras.layers.Lambda(lambda x: hub_layer(x), input_shape=[], dtype=tf.string),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

C:\Users\24677\anaconda3\envs\ml\lib\site-packages\tensorflow_hub\__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version



C:\Users\24677\anaconda3\envs\ml\lib\site-packages\keras\src\layers\core\lambda_layer.py:65: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [42]:
import tensorflow_hub as hub
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds

# --------------------------
# 1. 加载并预处理数据集
# --------------------------
datasets, info = tfds.load('imdb_reviews', as_supervised=True, with_info=True)
batch_size = 32

def preprocess(X, y):
    # 截断评论，减少计算量
    X = tf.strings.substr(X, 0, 300)
    return X, y

train_set = datasets['train'].batch(batch_size).map(preprocess).prefetch(1)

# --------------------------
# 2. 把 TF Hub 层包装成自定义层（关键修复！）
# --------------------------
class HubEmbeddingLayer(keras.layers.Layer):
    def __init__(self, hub_url, trainable=False):
        super().__init__()
        self.hub_layer = hub.KerasLayer(hub_url, trainable=trainable)

    def call(self, inputs):
        # 直接调用 hub_layer，适配 Keras 符号张量
        return self.hub_layer(inputs)

# 实例化自定义层
hub_url = "https://tfhub.dev/google/tf2-preview/nnlm-en-dim50/1"
embedding_layer = HubEmbeddingLayer(hub_url, trainable=False)

# --------------------------
# 3. 构建模型（和书上效果完全一致）
# --------------------------
model = keras.Sequential([
    # 用我们的自定义层，直接接收字符串输入
    keras.layers.Input(shape=[], dtype=tf.string),
    embedding_layer,
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
])

# --------------------------
# 4. 编译并训练
# --------------------------
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

history = model.fit(train_set, epochs=5)

Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.6464 - loss: 0.6300
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.6620 - loss: 0.6125
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.6668 - loss: 0.6083
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.6695 - loss: 0.6044
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6722 - loss: 0.6007


# 16.3 神经网络翻译的编码器-解码器网络

In [47]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

# 定义超参数
vocab_size = 10000
embed_size = 128
latent_dim = 512

# 1. 编码器
encoder_inputs = keras.Input(shape=[None], dtype=np.int32)
encoder_emb = keras.layers.Embedding(vocab_size, embed_size)(encoder_inputs)
encoder_lstm = keras.layers.LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_emb)
encoder_states = [state_h, state_c]

# 2. 解码器（训练时）
decoder_cell = keras.layers.LSTMCell(512)
output_layer = keras.layers.Dense(vocab_size)
decoder_inputs = keras.Input(shape=[None], dtype=np.int32)
decoder_emb = keras.layers.Embedding(vocab_size, embed_size)(decoder_inputs)
decoder_lstm = keras.layers.LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_emb, initial_state=encoder_states)
decoder_dense = keras.layers.Dense(vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# 3. 构建训练模型
model = keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)

# 编译模型
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 模型摘要
model.summary()

Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_10      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_11      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_6         │ (None, None, 128) │  1,280,000 │ input_layer_10[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_7         │ (None, None, 128) │  1,280,000 │ input_layer_11[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_4 (LSTM)       │ [(None, 512),     │  1,312,768 │ embedding_6[0][0] │
│                     │ (None, 512),      │            │                   │
│                     │ (None, 512)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_5 (LSTM)       │ [(None, None,     │  1,312,768 │ embedding_7[0][0… │
│                     │ 512), (None,      │            │ lstm_4[0][1],     │
│                     │ 512), (None,      │            │ lstm_4[0][2]      │
│                     │ 512)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, None,      │  5,130,000 │ lstm_5[0][0]      │
│                     │ 10000)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 10,315,536 (39.35 MB)

 Trainable params: 10,315,536 (39.35 MB)

 Non-trainable params: 0 (0.00 B)


### 模型基本流程

- **输入与输出**：英语句子送入编码器，解码器输出法语翻译。法语翻译同时作为解码器的输入，但向后偏移一步（即解码器当前步接收上一步应输出的单词，而非实际输出）。
- **起始与结束标记**：解码器第一个输入是序列开始（SOS）标记，希望最终输出序列结束（EOS）标记。

### 关键处理技巧

- **反转源句子**：在送入编码器前，将英语句子反转（例如“I drink milk”变为“milk drink I”）。这使句子开头（通常是解码器先需要翻译的部分）最后进入编码器，有利于翻译效果。
- **词嵌入**：每个单词由其ID表示，通过嵌入层转换为词向量，供编码器和解码器使用。
- **输出概率**：解码器每步输出词汇表（法语）中每个单词的分数，经softmax转为概率，选最高概率词作为输出。这等同于分类任务，损失函数可用`sparse_categorical_crossentropy`。

### 推理阶段（训练后）

- 没有目标句子提供给解码器，改为将上一步输出的单词作为当前步的输入（需经过嵌入查找表），如图16-4所示。

### 实现中的额外细节

1. **变长序列处理**：句子长度不一，而张量形状固定。可通过填充（padding）使所有序列等长，并需要屏蔽（masking）忽略填充部分。
2. **损失屏蔽**：EOS标记之后的输出不应贡献损失，例如输出“Je bois du lait <eos> oui”中“oui”的损失应忽略。
3. **大词汇量时的计算优化**：输出词汇量很大（如50,000）时，完整softmax计算量巨大。可使用**采样softmax**（sampled softmax）——只计算正确单词和随机负样本的对数，近似损失。训练时用`tf.nn.sampled_softmax_loss()`，推理时仍用普通softmax。
4. **TensorFlow Addons 示例代码**：提供了使用`tensorflow_addons`构建基本编码器-解码器的代码片段，其中使用了`TrainingSampler`（训练时使用目标嵌入作为输入）。另外还提到**ScheduledEmbeddingTrainingSampler**，它按一定概率在目标嵌入和实际输出嵌入间随机选择，并可随时间调整概率，有助于训练过渡。



## 16.3.1 双向RNN


### 常规RNN的局限
常规循环层在每个时间步只能查看**过去和当前**的输入，具有“因果关系”，无法看到未来的信息。这在时间序列预测中合理，但对于许多NLP任务（如神经机器翻译），理解一个词往往需要参考其**后续的上下文**。

### 示例说明
考虑两个短语：
- “the **Queen** of United Kingdom”
- “the **queen** of hearts”

单词“queen”的具体含义（指君主还是扑克牌中的Q）必须看到后面的词才能确定。若只从前往后读，无法准确判断。

### 双向RNN的原理
为了引入未来信息，可以同时运行两个循环层：
- 一个**从左至右**读取序列（正向）
- 一个**从右至左**读取序列（反向）

然后将两个方向在每个时间步的输出**组合**（通常是拼接/合并），得到既包含过去又包含未来上下文的表示。这就是**双向循环层**，如图16-5所示。

### Keras实现
在Keras中，使用 `keras.layers.Bidirectional` 层包裹一个循环层即可。例如：

```python
keras.layers.Bidirectional(keras.layers.GRU(10, return_sequences=True))
```

- `Bidirectional` 会创建该循环层的两个副本（一个正向，一个反向）
- 输出时，每个时间步将两个方向的输出拼接起来
- 如果原循环层有10个单元，双向后的输出维度为 **20**（每个时间步20个值）

> 注意：此处使用GRU举例，同样适用于LSTM等其他循环层。

## 16.3.2 集束搜索

## 16.3.2 集束搜索

### 问题背景

使用训练好的编码器-解码器模型进行翻译时，常见的做法是**贪婪解码**：每个时间步只选择概率最高的单词作为输出。这可能导致次优结果，因为模型无法回头修正之前的错误选择。

**示例**：翻译法语句子“Comment vas-tu”。期望输出“How are you?”，但模型可能输出“How will you?”。原因是在训练数据中存在类似“Comment vas-tu jouer?”（意为“How will you play?”）的句子，模型看到“Comment vas”后倾向于输出“How will”。一旦做出这一选择，后续就无法更正。

### 集束搜索原理

集束搜索（Beam Search）允许模型保留多个候选序列，并在后续步骤中动态调整。

- **集束宽度（beam width）**：参数 $ k $，表示每一步保留的最有可能的句子数量。
- **过程**：
  1. 第一步：模型输出所有单词的概率，保留概率最高的 $ k $ 个单词作为初始候选。
  2. 随后每一步：对当前 $ k $ 个候选句子中的每一个，计算所有可能的下一个单词的概率，得到 $ k \times V $ 个扩展句子（$ V $ 是词汇表大小），总概率为原句概率乘新词概率。然后只保留总概率最高的 $ k $ 个句子。
  3. 重复直到遇到句子结束标记（`<eos>`）或达到最大长度。

### 示例（$ k = 3 $）

翻译“Comment vas-tu”：

- **第1步**：模型输出各单词概率。假设前三位是：“How” 75%，“What” 3%，“You” 1%。短列表为 `["How", "What", "You"]`。
- **第2步**：对每个候选句子查找下一个单词。例如：
  - 以“How”开头的句子，模型预测“will”的概率为36%，则句子“How will”的概率 = 75% × 36% = 27%。
  - 同时，“are”的概率可能为50%，“How are”的概率 = 75% × 50% = 37.5%。
  计算所有 $ 3 \times 10{,}000 $ 个扩展句子的概率，保留前三名。此时“How will”可能还在，但“How are”也会保留。
- **第3步**：继续扩展。可能得到“How are you?”（10%）、“How do you?”（8%）和“How will you?”（2%）。
- **后续**：最终可能得到“How are you <eos>?”（6%）、“How do you do?”（7%）等。可以看到“How will”被淘汰，而模型保留了多个合理翻译。

> 集束搜索不需要额外训练，仅通过更智能的解码策略即可提升翻译质量。

### TensorFlow Addons 实现

使用 `BeamSearchDecoder` 包装解码器：

```python
beam_width = 10
decoder = tfa.seq2seq.beam_search_decoder.BeamSearchDecoder(
    cell=decoder_cell, beam_width=beam_width, output_layer=output_layer)

# 将编码器最终状态复制 beam_width 份
decoder_initial_state = tfa.seq2seq.beam_search_decoder.tile_batch(
    encoder_state, multiplier=beam_width)

outputs, _, _ = decoder(
    embedding_decoder,
    start_tokens=start_tokens,
    end_token=end_token,
    initial_state=decoder_initial_state)
```

**说明**：
- `BeamSearchDecoder` 内部管理多个解码器副本（数量等于集束宽度）。
- `tile_batch` 将编码器的最终状态复制多份，供每个副本使用。
- 传入开始和结束标记，解码器自动执行集束搜索。

> 注意：集束搜索在短句翻译上效果良好，但对长句效果仍不理想，主要受限于 RNN 有限的短期记忆。下一节将介绍**注意力机制**来解决这一问题。

In [49]:
def beam_search_step(probabilities, beam_width):
    """
    集束搜索的单步实现：从词表概率中，选概率最高的前beam_width个词
    """
    # 选概率最高的beam_width个词
    top_indices = probabilities.argsort()[-beam_width:][::-1]
    # 返回词ID和对应的概率
    return [(idx, probabilities[idx]) for idx in top_indices]

# 16.4 注意力机制



### 动机与基本思想

在基础的编码器-解码器模型中，输入单词（如“milk”）到其翻译（如“lait”）的路径很长，需要经过多个时间步。这导致RNN的短期记忆成为瓶颈，尤其在长句子（超过30个单词）上效果不佳。

**注意力机制**允许解码器在每个时间步**动态关注**编码器中与当前翻译最相关的输入单词。例如，当解码器需要输出“lait”时，它会将注意力集中在输入单词“milk”上，从而缩短信息路径，缓解短期记忆限制。

### 工作原理

- 解码器不再仅仅接收编码器的**最终隐藏状态**，而是接收**编码器所有输出**的**加权和**。
- 权重 $ \alpha_{i,j} $ 表示在第 $ i $ 个解码器时间步中，第 $ j $ 个编码器输出的重要程度。
- 加权和作为注意力向量输入解码器，与前一解码器隐藏状态以及前一目标单词共同参与当前步的计算。

### 对齐模型（注意力层）

权重 $ \alpha_{i,j} $ 并非手工设定，而是由一个**小型神经网络**（对齐模型，也称注意力层）生成，该网络与整个编码器-解码器模型联合训练。

**计算过程**（以 Bahdanau 注意力为例，见图16-6右侧）：
1. 一个**时间分布的Dense层**（单个神经元）接收：
   - 所有编码器输出
   - 解码器的**前一隐藏状态**（如 $ h_2 $）
2. 该层为每个编码器输出 $ j $ 计算一个**分数** $ e_{(i,j)} $，衡量该输出与解码器前一隐藏状态的对齐程度。
3. 所有分数通过 **softmax** 归一化，得到权重 $ \alpha_{(i,j)} $，且满足 $ \sum_j \alpha_{(i,j)} = 1 $。

> 这种特定的注意力机制称为 **Bahdanau 注意力**（或**合并注意力/加法注意力**），因其将编码器输出与解码器前一隐藏状态合并计算。

### 注意力权重的数学形式

公式16-1总结了三种常见的对齐分数计算方式：

$$
\tilde{h}_{(i)} = \sum_j \alpha_{(i,j)} y_{(j)}, \quad
\alpha_{(i,j)} = \frac{\exp(e_{(i,j)})}{\sum_j \exp(e_{(i,j)})}
$$



其中 $ e_{(i,j)} $ 可以有三种形式：
- **点积**：$ e_{(i,j)} = h_{(i,j)}^T y_{(j)} $
- **通用（带权重矩阵）**：$ e_{(i,j)} = h_{(i,j)}^T W y_{(j)} $
- **合并（Bahdanau风格）**：$ e_{(i,j)} = v^T \tanh(W [h_{(i,j)}; y_{(j)}]) $

### TensorFlow Addons 实现（Luong 注意力）

以下代码展示如何使用 TensorFlow Addons 将注意力添加到编码器-解码器模型：

```python
attention_mechanism = tfa.seq2seq.attention_wrapper.LuongAttention(
    units, encoder_state, memory_sequence_length=encoder_sequence_length)

attention_decoder_cell = tfa.seq2seq.attention_wrapper.AttentionWrapper(
    decoder_cell, attention_mechanism, attention_layer_size=n_units)
```

- 先创建注意力机制（此处为 Luong 注意力，另一种常见类型）
- 然后用 `AttentionWrapper` 包裹解码器单元，即可在解码过程中应用注意力。

---



## 16.4.1 视觉注意力



注意力机制不仅用于机器翻译，还被扩展到其他领域，**视觉注意力**是早期成功应用之一。

**任务**：图像标题生成（Image Captioning）
- 使用卷积神经网络（CNN）处理图像，输出**特征图**。
- 解码器RNN通过注意力机制在生成每个单词时，**关注图像的不同区域**。

**示例**（图16-7）：
- 生成标题“一个女人在公园里扔飞盘”
- 当输出单词“飞盘”时，模型的注意力集中在图像中飞盘的位置

> 该工作源于 Kelvin Xu 等人2015年的论文《Show, Attend and Tell: Neural Image Caption Generation with Visual Attention》。

---

**小结**：注意力机制通过动态对齐编码器输入和解码器状态，有效改善了长序列翻译质量，并成功推广到图像标题生成等跨模态任务。下一节将介绍更强大的 **Transformer** 架构。

## 16.4.2 Transformer架构


Transformer 是 2017 年 Vaswani 等人提出的革命性架构，完全抛弃了 RNN，仅依赖**注意力机制**（“Attention Is All You Need”）。其整体结构如图。
![Transformer 模型示意图](transformer.png)
### 架构概览

- **编码器（左侧）**：输入为单词 ID 序列（形状 `[batch, max_input_len]`），输出每个单词的 512 维表征（形状 `[batch, max_input_len, 512]`）。编码器重复堆叠 $N=6$ 次。
- **解码器（右侧）**：训练时接收**右移一位**的目标句子（开头插入 SOS 标记），同时接收编码器的输出。解码器也堆叠 6 次，输出每个位置的下一个单词的概率（形状 `[batch, max_output_len, vocab_size]`）。
- **推理时**：无法获得目标句子，改用上一步的输出作为当前输入（从 SOS 开始），逐步生成翻译。

### 常见组件

Transformer 包含多个已熟悉的模块：
- 两个嵌入层（输入嵌入、输出嵌入）
- $5 \times N$ 个跳过连接（残差连接），每个后跟一个层归一化（Add & Norm）
- $2 \times N$ 个前馈网络（FFN），每个由两个 Dense 层组成：第一个用 ReLU，第二个无激活函数
- 最终输出层为带 softmax 的 Dense 层
> 所有层都是**时间分布**的（每个单词独立处理），但这样会丢失序列顺序信息，因此需要引入**位置编码**。

### 位置编码

由于自注意力本身不包含位置信息，Transformer 通过向词嵌入添加**位置嵌入**来编码单词在句子中的位置。论文中使用了固定正弦/余弦函数（公式 16-2）：

$$
P_{p,2i} = \sin\left(\frac{p}{10000^{2i/d}}\right), \quad
P_{p,2i+1} = \cos\left(\frac{p}{10000^{2i/d}}\right)
$$

- $p$：位置索引，$i$：维度索引，$d$：嵌入维度。
- 每个维度的正弦/余弦频率不同，使模型能够轻松学习相对位置。
- 图 16-9 展示了位置嵌入矩阵的可视化。

**TensorFlow 实现**：自定义 `PositionalEncoding` 层，预先计算嵌入矩阵，在 `call()` 中加到输入嵌入上。

```python
class PositionalEncoding(keras.layers.Layer):
    def __init__(self, max_steps, max_dims, dtype=tf.float32, **kwargs):
        super().__init__(dtype=dtype, **kwargs)
        if max_dims % 2 == 1: max_dims += 1
        p, i = np.meshgrid(np.arange(max_steps), np.arange(max_dims // 2))
        pos_emb = np.empty((1, max_steps, max_dims))
        pos_emb[0, :, 0::2] = np.sin(p / 10000**(2 * i / max_dims)).T
        pos_emb[0, :, 1::2] = np.cos(p / 10000**(2 * i / max_dims)).T
        self.positional_embedding = tf.constant(pos_emb.astype(self.dtype))
    def call(self, inputs):
        shape = tf.shape(inputs)
        return inputs + self.positional_embedding[:, :shape[-2], :shape[-1]]
```

使用示例：
```python
embed_size = 512; max_steps = 500; vocab_size = 10000
encoder_inputs = keras.layers.Input(shape=[None], dtype=np.int32)
decoder_inputs = keras.layers.Input(shape=[None], dtype=np.int32)
embeddings = keras.layers.Embedding(vocab_size, embed_size)
encoder_embeddings = embeddings(encoder_inputs)
decoder_embeddings = embeddings(decoder_inputs)
positional_encoding = PositionalEncoding(max_steps, max_dims=embed_size)
encoder_in = positional_encoding(encoder_embeddings)
decoder_in = positional_encoding(decoder_embeddings)
```

### 多头注意力

Transformer 的核心是**多头注意力**（Multi-Head Attention），它建立在**缩放点积注意力**（Scaled Dot-Product Attention）之上。

#### 缩放点积注意力（公式 16-3）

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_{\text{keys}}}}\right) V
$$

- $Q$（查询）：形状 `[n_queries, d_keys]`
- $K$（键）：形状 `[n_keys, d_keys]`
- $V$（值）：形状 `[n_values, d_values]`（通常 `n_keys = n_values`）
- 输出形状 `[n_queries, d_values]`，每个查询得到值的加权和。
- 缩放因子 $1/\sqrt{d_{\text{keys}}}$ 防止 softmax 饱和，保持梯度。
- 可通过掩码（加非常大的负数）忽略某些键/值。

**在不同层中的应用**：
- **编码器自注意力**：$Q=K=V=$ 编码器输入（每个单词与句子中所有单词比较，包括自身）。
- **解码器掩码自注意力**：同样 $Q=K=V=$ 解码器输入，但使用**因果掩码**（causal mask）防止看到未来单词。
- **编码器-解码器注意力**：$K, V$ 来自编码器输出，$Q$ 来自解码器输出（解码器查询编码器信息）。

Keras 内置 `keras.layers.Attention` 实现了缩放点积注意力，支持批处理和掩码（`use_scale=True` 学习缩放参数，`causal=True` 应用因果掩码）。

#### 多头注意力架构（图 16-10）

多头注意力将多个缩放点积注意力层**并行**组合：
- 对 $Q, K, V$ 分别进行**线性变换**（无激活函数的 Dense 层），投影到不同的子空间。
- 每个子空间独立计算注意力（称为一个“头”）。
- 将所有头的输出**拼接**，再经过一次线性变换得到最终输出。

**直觉**：单词嵌入编码了多种特征（词性、时态、位置等）。单个注意力层只能一次查询所有特征。多个头允许模型将特征投影到不同子空间，每个头专注于某一类特征（如一个头关注动词，另一个头关注时态），从而提升表达能力。

#### 简单实现示意（忽略残差、归一化、前馈）

```python
Z = encoder_in
for _ in range(6):
    Z = keras.layers.Attention(use_scale=True)([Z, Z])  # 编码器自注意力
encoder_outputs = Z

Z = decoder_in
for _ in range(6):
    Z = keras.layers.Attention(use_scale=True, causal=True)([Z, Z])  # 掩码自注意力
    Z = keras.layers.Attention(use_scale=True)([Z, encoder_outputs])  # 编码器-解码器注意力

outputs = keras.layers.TimeDistributed(
    keras.layers.Dense(vocab_size, activation="softmax")
)(Z)
```

> 注：本简化代码仅为展示注意力层用法，完整的 Transformer 还需添加残差连接、层归一化和前馈网络。

### 补充说明

- 完整的 Transformer 有 6 层编码器 + 6 层解码器，每层包含：多头注意力 → Add & Norm → 前馈网络 → Add & Norm。
- 位置编码是 Transformer 能够处理序列顺序的关键。
- 在撰写时（本书出版时），TensorFlow 2 尚未内置 `MultiHeadAttention`，但官方教程提供了完整实现。

# 16.5 最近语言模型的创新


2018 年被称为 “NLP 的 ImageNet 时刻”，一系列突破性工作推动了自然语言处理的巨大进步。其核心趋势包括：从 LSTM 转向 Transformer、大规模自监督预训练、以及对预训练模型进行微调以适应下游任务。本节介绍了 2018–2019 年的几项关键创新。

### 1. ELMo（Embeddings from Language Models）

- **论文**：Matthew Peters 等人，2018
- **核心思想**：从深度**双向**语言模型的内部状态中学习**上下文相关的词嵌入**。
- **特点**：传统的词嵌入（如 Word2Vec、GloVe）为每个单词分配固定向量，无法区分多义词。ELMo 能根据上下文动态生成嵌入。例如，“queen” 在 “Queen of the United Kingdom” 和 “queen bee” 中的嵌入将不同。
- **方法**：使用双向 LSTM 训练语言模型，将各层隐藏状态加权组合作为词嵌入。

### 2. ULMFiT（Universal Language Model Fine-tuning）

- **论文**：Jeremy Howard 与 Sebastian Ruder，2018
- **核心思想**：验证了**无监督预训练 + 监督微调**在 NLP 任务中的强大有效性。
- **方法**：在大规模文本语料上使用**自监督学习**（预测下一个词）训练 LSTM 语言模型，然后在具体任务（如文本分类）上微调。
- **成果**：
  - 在 6 个文本分类任务上大幅超越先前最佳结果（错误率降低 18–24%）。
  - 仅用 100 个标注样本微调，就能达到从 10 000 个样本从头训练的性能。

### 3. GPT（Generative Pre-Training）

- **论文**：Alec Radford 等（OpenAI），2018
- **架构**：类似 Transformer 的**解码器**（仅使用掩码多头注意力，即因果注意力），堆叠 12 层。
- **方法**：在大规模语料上进行自监督语言模型预训练（预测下一个词），然后对多种下游任务进行微调，每个任务只需对模型做极小的结构调整。
- **任务类型**：包括文本分类、自然语言推理（判断句子 A 是否蕴含 B）、语义相似度、问答等。
- **意义**：展示了生成式预训练的强大迁移能力。

### 4. GPT-2

- **论文**：Alec Radford、Jeffrey Wu 等（OpenAI），2019 年 2 月
- **架构**：与 GPT 类似，但**规模巨大**——参数超过 15 亿（同时发布较小版本，1.17 亿参数，附带预训练权重，开源地址：https://github.com/openai/gpt-2）。
- **突破**：可以在**无需任何微调**的情况下，在多个任务上取得良好性能，称为**零次学习（Zero-Shot Learning, ZSL）**。

### 5. BERT（Bidirectional Encoder Representations from Transformers）

- **论文**：Jacob Devlin 等（Google），2019
- **架构**：类似于 Transformer 的**编码器**（没有掩码注意力，即**双向**），因此能同时利用左右上下文。BERT 名字中的 “B” 即代表 Bidirectional。
- **两个创新性的预训练任务**：

  **① 掩码语言模型（Masked Language Model, MLM）**
  - 句子中 15% 的词被随机选中处理：
    - 80% 的概率替换为 `[MASK]` 标记
    - 10% 的概率替换为随机词
    - 10% 的概率保持不变
  - 模型需要预测被选中的原始词（其他位置输出忽略）。
  - 这样的设计减少了预训练与微调之间的差异（微调时没有 `[MASK]` 标记）。

  **② 下一句预测（Next Sentence Prediction, NSP）**
  - 训练模型判断两个句子是否连续（是否为原文中的前后句）。
  - 示例：
    - “The dog sleeps.” 与 “It snores loudly.” → 连续（正例）
    - “The dog sleeps.” 与 “The Earth orbits the Sun.” → 不连续（负例）
  - 这项任务显著提升了模型在问答、推理等需要句子间关系的任务上的表现。

### 总结趋势

- **分词**：采用更合理的子词分词方法（如 BPE、WordPiece）。
- **架构**：从 LSTM 全面转向 Transformer（尤其是其编码器或解码器变体）。
- **训练范式**：自监督预训练（语言建模、掩码预测等） → 微调（或直接零次学习），成为主流。
- **未来展望**：虽然当时 Transformer 占据主导，但作者指出未来可能有新架构（如基于 CNN 的序列到序列模型，例如 Maha Elbayad 等人 2018 年的论文《Pervasive Attention: 2D Convolutional Neural Networks for Sequence-to-Sequence Prediction》）。